### Wikipedia Retriever
A Wikipedia Retriever is a retriever that queries the Wikipedia API to fetch relevant
content for a given query.<br>
**How It Works**
1. We give a query.
2. It sends the query to Wikipedia's API.
3. It retrieves the most relevant articles.
4. It returns them as LangChain Document objects.

In [1]:
from langchain_community.retrievers import WikipediaRetriever

In [2]:
retriever = WikipediaRetriever(
    top_k_results = 2 , lang = "en"
)

In [3]:
query = "The geopolitical history of bangladesh and india from the perspective of a pakistani."
# get the results
docs = retriever.invoke(query)

In [7]:
for i , doc in enumerate(docs):
    print(f"========================= Doc: {i+1} ======================")
    print(doc)
    print(" = " * 30)

========================= Doc: 1 ======================
page_content='The history of Bangladesh dates back over four millennia to the Chalcolithic period. The region's early history was characterized by a succession of Hindu and Buddhist kingdoms and empires that fought for control over the Bengal region. Islam arrived in the 8th century and gradually became the dominant religion from the early 13th century with conquests led by Bakhtiyar Khalji and the activities of Sunni missionaries, like Shah Jalal. Muslim rulers promoted the spread of Islam by building mosques across the region. From the 14th century onward, Bengal was ruled by the Bengal Sultanate, founded by Fakhruddin Mubarak Shah, who established an individual currency. The Bengal Sultanate expanded under rulers like Shamsuddin Ilyas Shah, leading to economic prosperity and military dominance, with Bengal being referred to by Europeans as the richest country to trade with. The region later became a part of the Mughal Empire, a

### Vector Store Retriever
A vector store retriever in LangChain is the common type of retriever that lets you
search and fetch documents from a vector store based on semantic similiarity using
vector embeddings.
**How It works**
1. Store the vector documents in a vector store.
2. Each document is converted into a dense vector using and embedding model.
3. When user enters a query:
   - It's also turned into a vector.
   - The retriever compares the query vector with the stored vectors.
   - It retrieves the top-k most similar ones.

In [8]:
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

In [9]:
documents = [
    Document(
        page_content="LangChain is a framework for building applications powered by large language models.",
        metadata={"source": "intro.txt", "page": 1}
    ),
    Document(
        page_content="It provides tools for chaining LLMs with external data sources like APIs and databases.",
        metadata={"source": "intro.txt", "page": 2}
    ),
    Document(
        page_content="Text splitting is a crucial step in preprocessing documents for embedding and retrieval.",
        metadata={"source": "preprocessing.txt", "page": 5}
    ),
    Document(
        page_content="Vector stores like Chroma allow efficient similarity search over embedded documents.",
        metadata={"source": "vector_db.txt", "page": 10}
    ),
    Document(
        page_content="Embeddings convert text into dense numerical vectors that capture semantic meaning.",
        metadata={"source": "embeddings.txt", "page": 3}
    ),
    Document(
        page_content="Retrievers are used to fetch relevant documents based on a query using similarity metrics.",
        metadata={"source": "retrieval.txt", "page": 7}
    )
]

In [10]:
from langchain_huggingface import HuggingFaceEmbeddings
model_name = "Qwen/Qwen3-Embedding-0.6B"
embedding_model = HuggingFaceEmbeddings(
  model_name = model_name
)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

In [11]:
# create the vector store
vectorstore = Chroma.from_documents(
    documents = documents,
    embedding = embedding_model,
    collection_name = "my_collection",
    persist_directory = "./samaple_collection"
)

In [12]:
# convert vector-store into a retriever
retriever = vectorstore.as_retriever(
    search_kwargs = {"k": 2}
) 

In [13]:
query = "What is Chroma used for?"
results = retriever.invoke(query)

In [14]:
for res in results:
    print("\n\n")
    print(res)




page_content='Vector stores like Chroma allow efficient similarity search over embedded documents.' metadata={'page': 10, 'source': 'vector_db.txt'}



page_content='LangChain is a framework for building applications powered by large language models.' metadata={'source': 'intro.txt', 'page': 1}


In [15]:
vectorstore.similarity_search(query = query , k = 2)

[Document(metadata={'page': 10, 'source': 'vector_db.txt'}, page_content='Vector stores like Chroma allow efficient similarity search over embedded documents.'),
 Document(metadata={'page': 1, 'source': 'intro.txt'}, page_content='LangChain is a framework for building applications powered by large language models.')]

In [ ]:
mmr_retriever = vectorstore.as_retriever(
    search_type = "mmr",
    search_kwargs = {
        "k" : 2, 
        "lambda_mult": 0.25 # relevance diversity balance(if low then don't include same doc)
    }
)
mmr_retriever.invoke(query)

[Document(metadata={'source': 'vector_db.txt', 'page': 10}, page_content='Vector stores like Chroma allow efficient similarity search over embedded documents.'),
 Document(metadata={'source': 'intro.txt', 'page': 1}, page_content='LangChain is a framework for building applications powered by large language models.')]

### Multi-Query Retriever

In [18]:
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [19]:
# Create FAISS vector store
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(documents=all_docs, embedding=embedding_model)

In [20]:
# Create retrievers
similarity_retriever = vectorstore.as_retriever(
    search_type="similarity", 
    search_kwargs={"k": 5}
)

In [21]:
from langchain_huggingface import HuggingFaceEndpoint , ChatHuggingFace
# define the model
llm = HuggingFaceEndpoint(
    repo_id = "meta-llama/Llama-3.1-8B-Instruct",
    task = "text-generation",
    temperature = 0.8,
    max_new_tokens = 2000
)
model = ChatHuggingFace(llm = llm)

In [23]:
from langchain_classic.retrievers import MultiQueryRetriever
multiQueryRetriever = MultiQueryRetriever.from_llm(
    retriever = vectorstore.as_retriever(
        search_kwargs = {'k': 5}
    ),
    llm = model
)

In [24]:
query = "How to improve energy levels and maintain balance?"

In [25]:
similarity_result = similarity_retriever.invoke(query)
multiquery_result = multiQueryRetriever.invoke(query)

In [26]:
for i , doc in enumerate(similarity_result):
    print(f"\n doc: {i}: \n {doc}")


 doc: 0: 
 page_content='Consuming leafy greens and fruits helps detox the body and improve longevity.' metadata={'source': 'H2'}

 doc: 1: 
 page_content='Drinking sufficient water throughout the day helps maintain metabolism and energy.' metadata={'source': 'H5'}

 doc: 2: 
 page_content='Mindfulness and controlled breathing lower cortisol and improve mental clarity.' metadata={'source': 'H4'}

 doc: 3: 
 page_content='Regular walking boosts heart health and can reduce symptoms of depression.' metadata={'source': 'H1'}

 doc: 4: 
 page_content='Deep sleep is crucial for cellular repair and emotional regulation.' metadata={'source': 'H3'}


In [27]:
for i , doc in enumerate(multiquery_result):
    print(f"\n doc: {i}: \n {doc}")


 doc: 0: 
 page_content='Python balances readability with power, making it a popular system design language.' metadata={'source': 'I2'}

 doc: 1: 
 page_content='Mindfulness and controlled breathing lower cortisol and improve mental clarity.' metadata={'source': 'H4'}

 doc: 2: 
 page_content='Consuming leafy greens and fruits helps detox the body and improve longevity.' metadata={'source': 'H2'}

 doc: 3: 
 page_content='Regular walking boosts heart health and can reduce symptoms of depression.' metadata={'source': 'H1'}

 doc: 4: 
 page_content='The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.' metadata={'source': 'I4'}

 doc: 5: 
 page_content='Drinking sufficient water throughout the day helps maintain metabolism and energy.' metadata={'source': 'H5'}

 doc: 6: 
 page_content='Deep sleep is crucial for cellular repair and emotional regulation.' metadata={'source': 'H3'}
